# Compare-Map Visualisations

This notebook loads a `metrics.jsonl` generated by `nas/compare-track.py`,
builds a table where rows are architectures and columns are maps, and then
plots the RMSEs so different planners can be compared at a glance.

Adjust the `RUN_DIR` value in the next cell if you want to point at a specific
run directory under `nas/compare-map/`. By default it selects the newest one.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

try:
    BASE_OUTPUT_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_OUTPUT_DIR = Path.cwd()


def latest_run_dir(base_dir: Path = BASE_OUTPUT_DIR) -> Path:
    candidates = sorted(
        [path for path in base_dir.glob("*/metrics.jsonl")],
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No metrics.jsonl files found under {base_dir}. Run nas/compare-track.py first."
        )
    return candidates[-1].parent

# RUN_DIR_OVERRIDE: Path | str | None = "compare_20260409T233950"  # e.g., "compare_20260410T133741"
RUN_DIR_OVERRIDE: Path | str | None = None


def resolve_run_dir() -> Path:
    if RUN_DIR_OVERRIDE:
        candidate = Path(RUN_DIR_OVERRIDE)
        if not candidate.is_absolute():
            candidate = BASE_OUTPUT_DIR / candidate
        if candidate.is_file():
            candidate = candidate.parent
        if not candidate.exists():
            raise FileNotFoundError(
                f"Override directory {candidate} was not found. Update RUN_DIR_OVERRIDE."
            )
        metrics_file = candidate / "metrics.jsonl"
        if not metrics_file.exists():
            raise FileNotFoundError(
                f"{metrics_file} was not found; choose a directory containing metrics.jsonl."
            )
        return candidate
    return latest_run_dir()

RUN_DIR = resolve_run_dir()
METRICS_FILE = RUN_DIR / "metrics.jsonl"
print(f"Using metrics from: {RUN_DIR}")

with METRICS_FILE.open("r", encoding="utf-8") as fh:
    METRIC_RECORDS = [json.loads(line) for line in fh if line.strip()]

if not METRIC_RECORDS:
    raise RuntimeError(f"{METRICS_FILE} is empty; rerun compare-track.")

rows: list[dict[str, object]] = []
for record in METRIC_RECORDS:
    map_name = record["map"]
    for run in record["runs"]:
        rows.append(
            {
                "map": map_name,
                "arch": run["label"],
                "rmse": run.get("rmse"),
            }
        )

df = pd.DataFrame(rows)
pivot = df.pivot_table(index="arch", columns="map", values="rmse")
pivot = pivot.sort_index()
print(f"Loaded RMSE table with shape {pivot.shape} (arches x maps).")


Using metrics from: /Users/zayahcortright/DAX.nosync/f1tenth_ng_zc/nas/compare-map/compare_20260410T133741
Loaded RMSE table with shape (8, 23) (arches x maps).


## RMSE table

Rows correspond to planner architectures; columns are maps. Each cell contains the
cross-track RMSE (in metres) reported by the simulator.


In [2]:
display(
    pivot.style
    .format("{:.4f}")
    .background_gradient(axis=0, cmap="YlGnBu")
    .set_properties(**{"text-align": "center"})
)


map,F1/Austin/Austin,F1/BrandsHatch/BrandsHatch,F1/Budapest/Budapest,F1/Catalunya/Catalunya,F1/Hockenheim/Hockenheim,F1/IMS/IMS,F1/Melbourne/Melbourne,F1/MexicoCity/MexicoCity,F1/Montreal/Montreal,F1/Monza/Monza,F1/MoscowRaceway/MoscowRaceway,F1/Nuerburgring/Nuerburgring,F1/Oschersleben/Oschersleben,F1/Sakhir/Sakhir,F1/SaoPaulo/SaoPaulo,F1/Sepang/Sepang,F1/Shanghai/Shanghai,F1/Silverstone/Silverstone,F1/Sochi/Sochi,F1/Spa/Spa,F1/Spielberg/Spielberg,F1/YasMarina/YasMarina,F1/Zandvoort/Zandvoort
arch,,,,,,,,,,,,,,,,,,,,,,,
arch1,0.1108,0.1060,0.1173,0.1197,0.0881,0.0483,0.1018,0.1136,0.1295,0.0846,0.1131,0.0803,0.0942,0.0961,0.0860,0.1033,0.1261,0.0813,0.0851,0.0805,0.0808,0.1115,0.0728
arch2,0.0934,0.0978,0.0986,0.1036,0.0791,0.0139,0.0944,0.1007,0.0959,0.0828,0.0947,0.0798,0.0857,0.0882,0.0749,0.0912,0.1272,0.0828,0.0679,0.0773,0.0761,0.0971,0.0816
arch3,0.0929,0.0604,0.0780,0.0870,0.0777,0.0137,0.0878,0.1127,0.0961,0.0720,0.1135,0.0775,0.0750,0.0862,0.0846,0.0911,0.1321,0.0711,0.0919,0.0564,0.0689,0.1200,0.0780
arch4,0.0963,0.0922,0.1111,0.1078,0.0896,0.0367,0.0887,0.1056,0.0713,0.0659,0.1103,0.0997,0.0923,0.0918,0.0980,0.1043,0.1333,0.0846,0.0875,0.0765,0.0800,0.1005,0.0882
arch5,0.1045,0.0834,0.0988,0.0991,0.0934,0.0445,0.0818,0.0998,0.0935,0.0693,0.1175,0.0985,0.1102,0.0868,0.0991,0.0994,0.1373,0.0920,0.0876,0.0834,0.0766,0.1056,0.0981
arch6,0.1001,0.0835,0.0968,0.0944,0.0917,0.0438,0.0784,0.0944,0.0993,0.0679,0.1120,0.0983,0.1125,0.0859,0.0977,0.0962,0.1256,0.0930,0.0881,0.0834,0.0788,0.0974,0.0983
arch7,0.0969,0.0844,0.1011,0.1012,0.0901,0.0458,0.0826,0.0996,0.0912,0.0697,0.1125,0.0995,0.1065,0.0881,0.0991,0.1021,0.1515,0.0900,0.0863,0.0813,0.0782,0.1035,0.0972
arch8,0.0889,0.0707,0.0834,0.0864,0.0852,0.0403,0.0705,0.0849,0.0755,0.0655,0.0989,0.0805,0.1091,0.0757,0.0873,0.0788,0.0969,0.0849,0.0793,0.0756,0.0710,0.0921,0.0938


## Connected RMSE comparison

A connected scatter plot (line plot with markers) makes it easy to see trends
per architecture: each line tracks an arch across maps, so you can spot where
one suddenly performs better or worse relative to the others. The x-axis
reflects each map explicitly so every point sits exactly on its label.


In [ ]:
maps = list(pivot.columns)
x_positions = range(len(maps))
highlight_arch = "arch8"
has_highlight = highlight_arch in pivot.index
fig, ax = plt.subplots(figsize=(14, 6))
other_arches = [arch for arch in pivot.index if not (has_highlight and arch == highlight_arch)]
if other_arches:
    muted_scale = np.linspace(0.3, 0.75, len(other_arches))
    for scale, arch in zip(muted_scale, other_arches):
        color = plt.cm.Greys(scale)
        ax.plot(
            x_positions,
            pivot.loc[arch].values,
            marker="o",
            linewidth=1.2,
            color=color,
            alpha=0.85,
            label=arch,
        )
if has_highlight:
    ax.plot(
        x_positions,
        pivot.loc[highlight_arch].values,
        marker="o",
        linewidth=2.5,
        color="#d62728",
        label=highlight_arch,
    )
ax.set_xticks(list(x_positions))
ax.set_xticklabels(maps, rotation=45, ha="right")
ax.set_xlim(-0.5, len(maps) - 0.5)
ax.set_ylabel("Cross-track RMSE (m)")
ax.set_xlabel("Map")
ax.set_title(f"RMSE comparison across maps ({RUN_DIR.name})")
ax.set_ylim(bottom=0)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
ax.legend(title="Architecture", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
